In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from imblearn.over_sampling import SMOTE
import joblib
from time import time
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier

# load dataset
project_root = Path.cwd()
data_path = None
for parent in [project_root] + list(project_root.parents):
    candidate = parent / 'Data' / 'Numerical' / 'Data' / '65_Nutrients_Data.csv'
    if candidate.exists():
        data_path = candidate
        project_root = parent
        break

if data_path is None:
    raise FileNotFoundError('Could not find 65_Nutrients_Data.csv under any parent of '+str(Path.cwd()))

print('Using data file:', data_path)
df = pd.read_csv(data_path)

# sanitize column names to remove special JSON chars (required by LightGBM)
df.columns = [re.sub(r'[^0-9a-zA-Z_]', '_', str(c)) for c in df.columns]
label_col = 'novaclass'
if label_col not in df.columns:
    raise KeyError(f"{label_col} column missing after sanitizing columns: {df.columns.tolist()}")

X = df.drop(columns=[label_col])
y = df[label_col].astype(int)
min_label = int(y.min())
if min_label != 0:
    y = y - min_label

# split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('train/test sizes', X_train.shape, X_test.shape)

# resample with SMOTE
sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X_train, y_train)
print('Resampled train sizes', X_res.shape, np.bincount(y_res))

# try imports for base estimators
estimators = []
try:
    from lightgbm import LGBMClassifier
    estimators.append(('lgbm', LGBMClassifier(random_state=42)))
except Exception as e:
    print('LightGBM not available:', e)
try:
    from catboost import CatBoostClassifier
    estimators.append(('cat', CatBoostClassifier(random_seed=42, verbose=0)))
except Exception as e:
    print('CatBoost not available, will fallback to HistGradientBoosting:', e)
    try:
        from sklearn.ensemble import HistGradientBoostingClassifier
        estimators.append(('hgb', HistGradientBoostingClassifier(random_state=42)))
    except Exception as e2:
        print('Fallback HGB not available:', e2)

if len(estimators) == 0:
    raise ImportError('No base estimators available (lightgbm/catboost/hgb)')

# stacking classifier with logistic regression meta-learner
stack = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression(max_iter=2000), n_jobs=-1, passthrough=False)

print('Training stacking ensemble with:', [name for name,_ in estimators])
t0 = time()
stack.fit(X_res, y_res)
t1 = time()

y_pred = stack.predict(X_test)
print('\nClassification report (stack):')
print(classification_report(y_test, y_pred))
print('Test f1_macro (stack):', f1_score(y_test, y_pred, average='macro'))
print('Train time (s):', round(t1-t0,2))

out_dir_stack = project_root / 'models' / 'numerical_65_stack' / 'smote'
out_dir_stack.mkdir(parents=True, exist_ok=True)
joblib.dump(stack, out_dir_stack / 'stack_65_numerical_smote.joblib')
print('Saved stacking model to', out_dir_stack)